# Data engineering and pre-processing

### agg_type, threshold, scaling, winsorize_std

### Imports

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import optuna 
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING) # Keep output clean
sys.path.append(os.path.abspath('../scripts'))

from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER

### Data

In [ ]:
synthetic_files = [
    "../data/synthetic/syn_a_ask_202601.parquet",
    "../data/synthetic/syn_a_bid_202601.parquet",
    "../data/synthetic/syn_b_ask_202601.parquet",
    "../data/synthetic/syn_b_bid_202601.parquet"
]

print("Building synthetic bars...")

builder = SPREAD(agg_type='volume', threshold=1000, active_hours=(0, 24))
df_synth = builder.build(synthetic_files)

print("Done. Ready for optimization.")

### Optuna

In [ ]:
def objective(trial):
    # --- 1. THE PARAMETERS OPTUNA WILL TUNE ---
    # Test 2 regimes (pure binary) vs 3 regimes (the "Garbage Bin" for fat tails)
    k_regimes = trial.suggest_categorical("k_regimes", [2, 3])
    
    # Test how aggressively we clip the data (None means no clipping)
    use_winsorization = trial.suggest_categorical("use_winsorize", [True, False])
    if use_winsorization:
        winsorize_std = trial.suggest_float("winsorize_std", 2.0, 5.0, step=0.5)
    else:
        winsorize_std = None
        
    # Test return scaling (helps the statsmodels solver converge)
    scaling = trial.suggest_categorical("scaling", [1000, 5000, 10000])
    
    # Static parameters for this block
    train_days = 5
    coint_window = 300
    z_window = 100

    try:
        # --- 2. RUN THE ENGINE ---
        live_trading_data, _ = ENGINE.walk_forward(
            df=df_synth,
            train_days=train_days,
            coint_window=coint_window,
            z_window=z_window,
            k_regimes=k_regimes,
            winsorize_std=winsorize_std,
            scaling=scaling,
            print_freq=999 # Silence the prints during optimization
        )
        
        # --- 3. RUN THE BACKTESTER ---
        bt = BACKTESTER(live_trading_data)
        results = bt.run(
            base_z=1.5, 
            exit_z=-0.25, 
            danger_threshold=0.80,   
            ar_limit=0.985,
            fee_bps=0.5
        )
        
        # --- 4. CALCULATE THE SCORE TO MAXIMIZE ---
        # We want to maximize the MS_AR Sharpe Ratio
        returns = results['Return_MS_AR'].fillna(0)
        if returns.std() == 0:
            return -99.0 # Penalize flatlines
            
        sharpe = (returns.mean() / returns.std()) * np.sqrt(252 * 24 * 60) # Approx annualized
        
        return sharpe

    except Exception as e:
        # If statsmodels fails to converge (common with bad params), prune the trial
        raise optuna.TrialPruned()

### Run

In [ ]:
# Create the study and run it for 50 iterations
study = optuna.create_study(direction="maximize", study_name="HMM_Tuning")

print("Starting Optuna Optimization...")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n=== BEST PARAMETERS FOUND ===")
print(f"Best Sharpe Ratio: {study.best_value:.2f}")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

### Visualising

In [ ]:
# Plot the optimization history
display(optuna.visualization.plot_optimization_history(study))

# Plot parameter importances (tells you which param actually mattered the most)
display(optuna.visualization.plot_param_importances(study))